# Silver Trade Incremental CDC

- **Purpose**: Transforms Bronze Trade data to Silver. Applies type casting, enriches with trade/status reference data, deduplicates keeping the latest `_ingest_ts`, and merges into the target.
- **Business Context**: PWG Pipeline - Trade Domain (Must run after reference tables).
- **Execution Frequency**: Per Batch
- **Inputs**: `bronze.trade`, `silver.tradetype`, `silver.statustype`
- **Outputs**: `silver.trade` (MERGE pattern)

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# Creating widgets here for the ease of coding and reusability
dbutils.widgets.text("catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id", "1")
catalog = dbutils.widgets.get("catalog")
batch_id = dbutils.widgets.get("batch_id")

bronze_trade_tbl = f"{catalog}.bronze.trade"
silver_trade_tbl = f"{catalog}.silver.trade"
silver_tt_tbl = f"{catalog}.silver.tradetype"
silver_st_tbl = f"{catalog}.silver.statustype"

In [0]:
# Importing functions and libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# spark.read.table(silver_trade_tbl).display( )

In [0]:
# Logging functions for initial load
l_df = spark.sql(f"SELECT * FROM {bronze_trade_tbl} ORDER BY _ingest_ts DESC LIMIT 1")
carried_run_id = str(l_df.select("_run_id").first()[0])

log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_trade', 'Starting processing for standalone silver trade CDC updates')
start_pipeline_run(spark, carried_run_id, batch_id)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'RUNNING')

In [0]:
# reading bronze table into dataframe
s_df = spark.read.table(bronze_trade_tbl).filter(col("_batch") == batch_id)
s_df.cache() # this will help cache the df so source count will not be recomputed

# Storing the initial count
source_count = s_df.count()

In [0]:
try:
    # Transformations logic here
    s_df1 = s_df.withColumns({
        "T_ID"      : col("T_ID")   .cast("bigint"),
        "T_CA_ID"   : col("T_CA_ID").cast("bigint"),
        "T_QTY"     : col("T_QTY")  .cast("int"),
        "T_CHRG"    : col("T_CHRG") .cast("decimal(10,2)"),
        "T_COMM"    : col("T_COMM") .cast("decimal(10,2)"),
        "T_TAX"     : col("T_TAX")  .cast("decimal(10,2)"),
        "T_IS_CASH" : col("T_IS_CASH").cast("boolean"),
        "T_DTS"     : to_timestamp(col("T_DTS"), "yyyy-MM-dd HH:mm:ss"),
        "T_BID_PRICE": col("T_BID_PRICE").cast("decimal(8,2)"),
        "T_TRADE_PRICE" : col("T_TRADE_PRICE").cast("decimal(8,2)")
    })
except Exception as e:
    print(f"Error during transformation: {e}")


In [0]:
try:
    # Reading refrence files in data frame
    tt_df = spark.read.table(silver_tt_tbl).select("TT_ID","TT_NAME","TT_IS_SELL","TT_IS_MRKT")
    st_df = spark.read.table(silver_st_tbl).select("ST_ID","ST_NAME")

    # Joining refrence files for enrichment of the table, using only selected columns i dont think metadata cols are needed
    s_df2 = (
        s_df1
        .join(tt_df, s_df1.T_TT_ID == tt_df.TT_ID, "left")
        .join(st_df, s_df1.T_ST_ID == st_df.ST_ID, "left")
    )
    
except Exception as e:
    print(f"Error during reference enrichment: {e}")


In [0]:
try:
    # Deduplication logic here which will handle duplicate recodes based on injestion timestamp
    s_df2.createOrReplaceTempView("trade_view")

    s_df3 = (
        spark.sql("""
                  select *, row_number() over( partition by T_ID order by _ingest_ts desc ) as rn 
                  from trade_view
                  """)
        .filter(col("rn") == 1)
        .drop('rn','_source_file','_ingest_ts')
        .withColumn("_load_ts", current_timestamp())
    )

    s_df3.createOrReplaceTempView("trade_view_final")
except Exception as e:
    print(f"Error during deduplication: {e}")

In [0]:
# Final merge logic here
try:
    (
        s_df3.limit(0).write
        .format("delta")
        .mode("ignore") # in this mode "ignore" means do nothing if the table already exists
        .saveAsTable(silver_trade_tbl)
    )

    spark.sql(f"""
                merge into {silver_trade_tbl} as t 
                using trade_view_final as s 
                on t.T_ID = s.T_ID 
                when matched then update set *
                when not matched then insert *
                """)
except Exception as e:
    print(f"Error during final merge: {e}")
            
# Finally reading final table and displaying the final count
df_final = spark.read.table(silver_trade_tbl)
target_count = df_final.count()

# spark.sql(f"describe history {silver_trade_tbl}").select("operationMetrics").limit(1).display() i will try this later
print(f"The final row count of the table is :{target_count}")

In [0]:
null_count = spark.sql(f"SELECT COUNT(*) FROM {silver_trade_tbl} WHERE T_ID IS NULL").first()[0]

log_dq_result(spark, carried_run_id, "silver.trade", "Null T_ID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_trade', 'Successfully completed standalone trade updates')

In [0]:
try:
    # Operations Logging
    # Extract the carry-forwarded _run_id from dataframe
    carried_run_id = str(df_final.select("_run_id").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id=batch_id,
        domain="TRADE",
        table_name="trade",
        source_layer="bronze",
        target_layer="silver",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=batch_id,
        layer="silver",
        table_name="trade",
        operation="MERGE",
        rows_affected=int(target_count)
    )

    print("Done")
except Exception as e:
    print(f"Error during operations logging: {e}")